In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader, Dataset
import cv2
import numpy as np
from skimage.color import rgb2lab, lab2rgb
import os

In [ ]:
class ChromaGAN_Generator(nn.Module):
    def __init__(self):
        super().__init__()

        # === ENCODER: VGG16 pretrained backbone ===
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        features = list(vgg.features.children())

        self.enc1 = nn.Sequential(*features[0:5])    # 256x256 -> 128x128, 64ch
        self.enc2 = nn.Sequential(*features[5:10])   # 128x128 -> 64x64,  128ch
        self.enc3 = nn.Sequential(*features[10:17])  # 64x64   -> 32x32,  256ch
        self.enc4 = nn.Sequential(*features[17:24])  # 32x32   -> 16x16,  512ch
        self.enc5 = nn.Sequential(*features[24:])    # 16x16   -> 8x8,    512ch

        # Adapt first layer: VGG expects 3 channels, we give 1 (grayscale)
        self.enc1[0] = nn.Conv2d(1, 64, kernel_size=3, padding=1)

        # === DECODER: Upsampling + Skip Connections ===
        self.dec5 = self._decoder_block(512, 512)
        self.dec4 = self._decoder_block(512 + 512, 256)  # +512 from skip
        self.dec3 = self._decoder_block(256 + 256, 128)  # +256 from skip
        self.dec2 = self._decoder_block(128 + 128, 64)   # +128 from skip
        self.dec1 = self._decoder_block(64  + 64,  32)   # +64  from skip

        # Final output: 2 channels (a and b)
        self.final = nn.Conv2d(32, 2, kernel_size=1)
        self.tanh  = nn.Tanh()  # output range [-1, 1] matches our ab normalization

    def _decoder_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Encode
        e1 = self.enc1(x)   # 64ch
        e2 = self.enc2(e1)  # 128ch
        e3 = self.enc3(e2)  # 256ch
        e4 = self.enc4(e3)  # 512ch
        e5 = self.enc5(e4)  # 512ch — bottleneck

        # Decode with skip connections
        d5 = self.dec5(e5)
        d4 = self.dec4(torch.cat([d5, e4], dim=1))
        d3 = self.dec3(torch.cat([d4, e3], dim=1))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        d1 = self.dec1(torch.cat([d2, e1], dim=1))

        return self.tanh(self.final(d1))  # shape: (B, 2, H, W)

In [ ]:
class ChromaGAN_Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: L channel (1ch) concatenated with ab channels (2ch) = 3ch total
        self.model = nn.Sequential(
            nn.Conv2d(3, 64,  kernel_size=4, stride=2, padding=1),  # 128x128
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64,  128, kernel_size=4, stride=2, padding=1), # 64x64
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), # 32x32
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1), # 31x31
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1,   kernel_size=4, stride=1, padding=1), # 30x30
            # No sigmoid — we use BCEWithLogitsLoss which includes it numerically
        )

    def forward(self, L, ab):
        # Concatenate L and ab so discriminator sees the full image context
        x = torch.cat([L, ab], dim=1)
        return self.model(x)

In [ ]:
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        # Use first 16 layers (up to relu3_3) for feature extraction
        self.features = nn.Sequential(*list(vgg.features)[:16])
        for param in self.features.parameters():
            param.requires_grad = False  # Freeze VGG — we only use it as a judge

    def forward(self, pred_ab, target_ab, L):
        # Convert LAB predictions back to 3-channel for VGG
        pred_rgb  = self._lab_to_3ch(L, pred_ab)
        target_rgb = self._lab_to_3ch(L, target_ab)

        pred_features   = self.features(pred_rgb)
        target_features = self.features(target_rgb)
        return F.mse_loss(pred_features, target_features)

    def _lab_to_3ch(self, L, ab):
        # Denormalize and stack into a 3-channel tensor for VGG input
        L_  = (L  + 1.0) * 50.0       # [-1,1] -> [0,100]
        ab_ = ab * 110.0               # [-1,1] -> [-110,110]
        lab = torch.cat([L_, ab_], dim=1)
        # VGG expects values roughly in [0,1] — use L as a proxy channel
        return lab / 100.0             # rough normalization for VGG

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

G = ChromaGAN_Generator().to(device)
D = ChromaGAN_Discriminator().to(device)
perceptual_loss_fn = VGGPerceptualLoss().to(device)

optimizer_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

adversarial_loss = nn.BCEWithLogitsLoss()
l1_loss          = nn.L1Loss()

dataset    = ChromaGANDataset("photos")
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

EPOCHS = 20
lambda_perceptual = 10.0   # weight for perceptual loss
lambda_l1         = 100.0  # weight for L1 pixel loss (added for stability)

for epoch in range(EPOCHS):
    for L_batch, ab_batch in dataloader:
        L_batch  = L_batch.to(device)
        ab_batch = ab_batch.to(device)

        batch_size = L_batch.size(0)
        real_label = torch.ones(batch_size, 1, 30, 30).to(device)
        fake_label = torch.zeros(batch_size, 1, 30, 30).to(device)

        # ── TRAIN DISCRIMINATOR ──────────────────────────────────────
        optimizer_D.zero_grad()

        # Real: D should output 1 for real (L, real_ab) pairs
        real_pred = D(L_batch, ab_batch)
        loss_D_real = adversarial_loss(real_pred, real_label)

        # Fake: D should output 0 for (L, generated_ab) pairs
        fake_ab = G(L_batch).detach()  # detach: don't backprop into G here
        fake_pred = D(L_batch, fake_ab)
        loss_D_fake = adversarial_loss(fake_pred, fake_label)

        loss_D = (loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        optimizer_D.step()

        # ── TRAIN GENERATOR ─────────────────────────────────────────
        optimizer_G.zero_grad()

        fake_ab   = G(L_batch)
        fake_pred = D(L_batch, fake_ab)

        # Loss 1: G wants D to think its output is real
        loss_G_adv = adversarial_loss(fake_pred, real_label)

        # Loss 2: Perceptual — feature-space similarity to ground truth
        loss_G_perceptual = perceptual_loss_fn(fake_ab, ab_batch, L_batch)

        # Loss 3: L1 pixel — low-frequency color accuracy
        loss_G_l1 = l1_loss(fake_ab, ab_batch)

        # Combined generator loss (ChromaGAN formula)
        loss_G = loss_G_adv + lambda_perceptual * loss_G_perceptual + lambda_l1 * loss_G_l1
        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"D: {loss_D.item():.4f} | "
          f"G_adv: {loss_G_adv.item():.4f} | "
          f"G_perc: {loss_G_perceptual.item():.4f} | "
          f"G_l1: {loss_G_l1.item():.4f}")

In [ ]:
def colorize_chromagan(image_path, G, device, size=256):
    G.eval()
    original_bgr = cv2.imread(image_path)
    gray         = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
    gray_rgb     = cv2.cvtColor(cv2.resize(gray, (size, size)), cv2.COLOR_GRAY2RGB)

    lab = rgb2lab(gray_rgb).astype("float32")
    L   = lab[:, :, 0] / 50.0 - 1.0
    L_tensor = torch.tensor(L, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_ab = G(L_tensor).squeeze(0).permute(1, 2, 0).cpu().numpy()

    pred_ab  = pred_ab * 110.0
    orig_L   = (L + 1.0) * 50.0

    result_lab        = np.zeros((size, size, 3), dtype="float32")
    result_lab[:,:,0] = orig_L
    result_lab[:,:,1:]= pred_ab

    result_rgb = (lab2rgb(result_lab) * 255).astype(np.uint8)
    return original_bgr, gray, result_rgb

# Run
original, gray, result = colorize_chromagan("photos/your_image.jpg", G, device)

import matplotlib.pyplot as plt
plt.figure(figsize=(14, 5))
plt.subplot(1,3,1); plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB)); plt.title("Original"); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(gray, cmap='gray'); plt.title("Grayscale Input"); plt.axis('off')
plt.subplot(1,3,3); plt.imshow(result); plt.title("Method 4: ChromaGAN"); plt.axis('off')
plt.savefig("method4_result.png", bbox_inches='tight', dpi=150)
plt.close()
print("Saved method4_result.png")